# PANOPLY Workbench Startup Notebook

This notebook walks you through uploading, validating, and configuring PANOPLY proteogenomic
data on a **Manifold** workbench, and produces two files:

* **`master-parameters.yaml`** -- default pipeline parameters, merged with the groups, colors,
  and toggles you choose below. Same shape as the parameters file produced by the Terra-based
  [`PANOPLY-startup-notebook.ipynb`](./PANOPLY-startup-notebook.ipynb).
* **`inputs.json`** -- a Cromwell-style input file (S3 file-paths + key parameters) for a PANOPLY
  WDL workflow -- `panoply_unified_workflow` by default, or any workflow in the
  [PANOPLY GitHub repo](https://github.com/broadinstitute/PANOPLY).

-----
### Using this notebook
1. This notebook requires an **R kernel** (IRkernel). If your environment only offers a Python
   kernel, install one first from a terminal, e.g. `mamba install -c conda-forge r-irkernel`,
   then select the R kernel for this notebook.
2. Run the **Setup** cell once per session -- it installs any R packages this notebook needs
   that aren't already present. A couple are large Bioconductor packages used for gene/protein-ID
   conversion, so the very first run can take a while; subsequent runs are fast.
3. This notebook works in terms of **sessions** -- see the *Sessions* section right after Setup
   for what that means and how to resume, restart, or reload one.
4. Run cells top to bottom the first time. Several cells prompt for input in the console below
   the cell -- read the preceding text before running each one.

### Prepare your data
Place (or upload) the following into `~/workbench/inputs/`:
* At least **one proteomics dataset** (global proteome, phosphoproteome, acetylome, and/or
  ubiquitylome), in [GCT v1.3 format](https://clue.io/connectopedia/gct_format).
* Genomics data -- CNA and/or RNA, also GCT v1.3.
* An `Annotation` CSV with at least `Sample.ID` and `Type` columns, plus any other sample
  annotations you have.
* Optionally: a `groups` CSV (one annotation-column name per line), your own parameter YAML
  (otherwise PANOPLY's default `master-parameters.yaml` is used), and/or PTM-SEA / GSEA `gmt`
  pathway databases (otherwise bundled defaults are used).

These are your **originals** -- the notebook copies them into your session and only ever
modifies the copies, so you can always come back to exactly what you uploaded.

Sample IDs (GCT column names) must match the `Sample.ID`s in your annotation table. See the
[PANOPLY wiki](https://github.com/broadinstitute/PANOPLY/wiki) for more on data formats.


### Setup
Run once per session. `wb_setup()` installs any R packages this notebook needs that aren't
already present (a couple are large Bioconductor packages used for gene/protein-ID conversion,
so the very first run can take a while; subsequent runs are fast), then loads the helper module.

In [ ]:
source("workbench-src/config.r")
wb_setup()

### Sessions
Everything you do below -- mapped files (and their validated/fixed copies), subsets, and
`master-parameters.yaml` (once you've built it) -- lives inside a **session** folder, so a
session is a complete, self-contained snapshot of one run; loading a saved session restores
all of it. `inputs.json` (built later, once you've named and saved a session -- see
*Finalized and Run Session* near the end) is the one exception: it's written directly into
that named session and is never copied back into `current-session`, even when you load that
session again later -- re-run *Generate `inputs.json`* after loading if you need it
regenerated.

* **`current-session`** is the one you're actively working in. It's always safe to keep
  editing -- nothing here is final until you explicitly name and save it (see *Finalized and
  Run Session* near the end).
* Once you've named and saved a session (e.g. `odg-v4`), you can come back to it later and
  pick "load a saved session" below -- this restores it into `current-session` so you can
  keep working from exactly that point.

Run the cell below to choose: resume `current-session` (pick up where you left off), start a
new one (reset `current-session` and begin fresh), or load a previously saved one.

In [ ]:
state <- wb_load_state()

#### Just need to regenerate `inputs.json` for an existing saved session?
The cell above fully copies a saved session into `current-session/`, which is slow for large
GCTs -- and unnecessary if all you want is a fresh `inputs.json` (that's always written
directly into the named session anyway, regardless of `current-session/`). Instead, run
`state <- wb_open_saved_session()` below, then skip straight down to the *Generate
`inputs.json`* section near the end -- you don't need to re-run anything in between.

Only use `state` from this path to regenerate `inputs.json` -- don't map new inputs, create
subsets, edit groups/colors, or rebuild `master-parameters.yaml` with it, since those all write
into `current-session/`, which this intentionally leaves untouched (and unrelated to this saved
session). For any of that, use the cell above instead.

In [ ]:
# state <- wb_open_saved_session()

### Configuration
Two things you may want to change:
* `GITHUB_REF` -- the branch/tag of [broadinstitute/PANOPLY](https://github.com/broadinstitute/PANOPLY)
  that workflow WDLs and the default `master-parameters.yaml` are fetched from. Defaults to the
  `issue-githubWDL` branch for now; bump this once a release branch is the intended target.
* `TARGET_WORKFLOW` -- which PANOPLY workflow to build `inputs.json` for. Defaults to
  `panoply_unified_workflow`. Run `wb_list_github_workflows()` (see the *Generate inputs.json*
  section below) to see other options.

In [ ]:
GITHUB_REF      <- "issue-githubWDL"
TARGET_WORKFLOW <- "panoply_unified_workflow"

state$github_ref      <- GITHUB_REF
state$target_workflow <- TARGET_WORKFLOW
state <- wb_save_state(state)

# Inputs
Run the cell below to map each file you've placed in `~/workbench/inputs/` to a data category
-- the available categories are listed for you as part of the prompt (a suggestion is offered
for filenames that look like they match one, e.g. `...-proteome-....gct`; press Enter to accept
it or type a different number). Each category can hold only one file; mapping a second file to
the same category overwrites the first.

If you have a single ZIP file instead of individual files, just place it in `~/workbench/inputs/`
too -- it's detected automatically and you'll be asked whether to unzip and use it, then whether
to also map any other loose files in that folder.

In [ ]:
state <- wb_load_and_map_inputs(state)

# Validation
Checks the annotation table for required columns and unique `Sample.ID`s, checks sample-ID
overlap between the annotation table and every mapped GCT file, and checks (or interactively
helps you fix) the gene-ID column in each proteomics/genomics GCT file.

In [ ]:
state <- wb_validate_inputs(state)

# Data Processing
Toggles proteomics normalization/filtering, and (if the relevant data was mapped above) PTM-SEA
and MetaboAnalyst. If you opt into PTM-SEA or MetaboAnalyst, you'll be prompted to confirm or
select the relevant ID column.

In [ ]:
state <- wb_select_preprocessing_options(state)

# COSMO Label Selection (optional)
COSMO (COrrection of Sample Mislabeling by Omics) needs 1-3 clinical attributes that are binary,
well-balanced, and free of NAs. You'll be shown the valid candidates from your annotation table
and asked to pick from among them.

In [ ]:
state <- wb_select_cosmo_attributes(state)

# Clumps-PTM Setup (optional)
Only offered if at least 2 of {phosphoproteome, acetylome, ubiquitylome} were mapped above.
Requires a reference FASTA (matching the accession-ID type used in your PTM data). If you
mapped one to the `clumpsFASTA` category in the *Inputs* section above, it's used automatically
-- otherwise you'll be prompted for its local path below (e.g. `~/workbench/inputs/reference.fasta`
or `workbench/inputs/reference.fasta`; `s3://` paths aren't accepted here -- upload the file
locally and provide that path instead).

In [ ]:
state <- wb_select_clumpsptm_groups(state)

# Groups
Groups are the categorical annotations used for association and enrichment analysis. By default,
all valid annotation columns are used (or the columns listed in your `groups` file, if you
provided one) -- pass an explicit `columns = c(...)` to override. Annotations with more than
`max_categories` unique values are excluded (or treated as continuous, if numeric).

In [ ]:
wb_list_annotation_columns(state)

In [ ]:
# state <- wb_select_groups(state, columns = c("Type", "Stage"), max_categories = 10)
state <- wb_select_groups(state, max_categories = 10)

## Color Schemes (optional)
### See the current color scheme

In [ ]:
wb_show_colors(state)

### Reset colors to defaults
Colors are assigned automatically based on the number of unique values per group; NA is always grey.

In [ ]:
state <- wb_reset_colors(state)

### Edit a color
Interactively pick a group, then either set one value's color at a time or replace every color
for that group at once from a comma-separated hex list. Repeats until you type 'quit'.

In [ ]:
state <- wb_edit_color(state)

# Sample Subsets
Replaces Terra sample sets: each subset is a local folder under `current-session/subsets/<name>/`
containing the GCT/CSV files filtered down to the matching samples. An `all` subset (every
sample) is always included; you'll then be asked whether to define additional subsets by
picking an annotation column, the value(s) to include, and a name -- repeat as many times as
you like. Nothing is actually written until you're done answering prompts, since writing GCTs
can take a while -- every subset you defined (including `all`) is then created in one batch.

In [ ]:
state <- wb_create_subset(state)

# Finalize Parameters
Builds `master-parameters.yaml`: PANOPLY's default module parameters (fetched from
`GITHUB_REF`, or your own uploaded parameter file, if you provided one), merged with the
groups/colors/toggles chosen above. Unlike `inputs.json`, nothing in this file's own content
is a file path, so it's built directly in `current-session/` here, alongside your subsets --
no named session required yet. It still needs to be carried into a named session (via the
*Finalized and Run Session* step below) before *Generate `inputs.json`* can find it, since
that step references this file by its own S3 path.

In [ ]:
master_params_path <- wb_build_master_parameters_yaml(state)
master_params_path

# Finalized and Run Session
The next cells will set up a finalized, *named* session (`sessions/<name>/`); this named 
session will have stable filepaths, a finalized YAML file with defaults, and an input-json for
the selected WDL.

## Name and Save This Session
`inputs.json` (built below) references files by their `s3://` path, and those paths get baked
into whatever Cromwell job you submit. If you kept working in `current-session` after
submitting a job, later edits (or starting a new session) could silently change or remove the
files that job is still using.

Naming and saving your session avoids this: it snapshots everything so far (mapped files,
subsets, and `master-parameters.yaml` if you've already built it above) into its own folder,
and generating `inputs.json` requires a named session for exactly this reason -- it's always
written inside it, never into the still-mutable `current-session`. If you go back and change
something afterward (e.g. add another subset, or rebuild `master-parameters.yaml`), just
re-run this cell with the same name to refresh the snapshot before regenerating anything.

In [ ]:
state <- wb_save_session(state)

## Workflow Run Options
Any remaining top-level toggles `TARGET_WORKFLOW` requires that aren't already inferred from
your data above (PTM-SEA, MetaboAnalyst, Clumps-PTM, normalization/filtering) -- you'll be
prompted for each one its WDL actually declares as required, so this adapts automatically if
`TARGET_WORKFLOW` is changed to something other than `panoply_unified_workflow`.

In [ ]:
state <- wb_select_workflow_toggles(state)

## Generate `inputs.json`
Builds a Cromwell-style `inputs.json` for `TARGET_WORKFLOW`, with every -omics/groups/database
file path translated to its `s3://` location and the parameters set above filled in. Uninvolved
inputs (e.g. per-task memory/disk overrides) are intentionally left for `master-parameters.yaml`
or Cromwell defaults to handle. You'll be prompted to pick which subset to build for.

Re-run this cell any time you want `inputs.json` for a different subset, or to refresh it after
a later change. File-path inputs (and `job_id`) are always refreshed, since those are recomputed
from the subset/session every time. Toggles/parameters (e.g. from *Workflow Run Options* above)
are different -- you'll be asked whether to refresh those too, since there's no way to tell
whether you've re-run that step with a genuine change versus having since hand-edited one of
those same keys directly in the JSON; answering no leaves every toggle/parameter untouched. If
one already exists at the standard path, you'll be offered the choice to update it in place or
point at a different copy instead (e.g. one you moved, renamed, or have been hand-editing); a
`.bak` backup is written before every update.

To target a different workflow, see what's available and update `TARGET_WORKFLOW` in the
*Configuration* section above:
```r
# wb_list_github_workflows()
```

In [ ]:
inputs_path <- wb_update_inputs_json_for_subset(state)
inputs_path

## Done
The following `inputs.json` file can be used to run the PANOPLY workflow selected above. Feel free to edit it with additional parameters before runtime; if the JSON needs to be regenerated for a new subset, only filepaths and parameters set within the notebook will be overwritten.

In [ ]:
cat("inputs.json\n  local:", inputs_path, "\n  s3:   ", wb_local_to_s3(inputs_path), "\n")